In [ ]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import sys
import boto3 
from functools import reduce 
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/02 15:46:26 WARN Utils: Your hostname, Admin-PC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/02 15:46:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/02 15:46:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
class DataLoader :
    def __init__(selt) :
        selt.spark = SparkConfig.create_sparksession() 
    def clean_works() :    
        s3 = boto3.client("s3")
        response = s3.list_objects_v2(Bucket = "mhai-bk" , Prefix = "raw_data/works/")
        files = [f"s3://mhai-bk/{item['Key']}" for item in response['Contents']]
        list_file = []
        list_col = []
        for file in files :
            works = spark.read.format("parquet").load(file) 
            
            works = works.withColumn("cover_id" , col("cover_id").cast("long"))
            for c in works.columns :
                if c not in list_col :
                    list_col.append(c)
            for c in list_col :
                if c not in works.columns :
                    works = works.withColumn(c , lit(None).cast("string"))
            list_file.append(works)
        works = reduce(lambda a,b : a.union(b) , list_file)

        works.limit(5).coalesce(1).write.mode("overwrite").parquet("s3://mhai-bk/data/works/")
    def clean_editions() :    
        s3 = boto3.client("s3")
        response = s3.list_objects_v2(Bucket = "mhai-bk" , Prefix = "raw_data/editions/")
        files = [f"s3://mhai-bk/{item['Key']}" for item in response['Contents']]
        list_file = []
        list_col = []
        for file in files :
            editions = spark.read.format("parquet").load(file) 
            if "number_of_pages" in editions.columns :
                editions = editions.withColumn("number_of_pages" , col("number_of_pages").cast("long"))
            
            for c in editions.columns :
                if c not in list_col :
                    list_col.append(c)
            for c in list_col :
                if c not in editions.columns :
                    editions = editions.withColumn(c , lit(None).cast("string"))
            list_file.append(editions)
        editions = reduce(lambda a,b : a.union(b) , list_file)
        editions.limit(5).coalesce(1).write.mode("overwrite").parquet("s3://mhai-bk/data/editions/")
    def clean_works() :        
        s3 = boto3.client("s3")
        response = s3.list_objects_v2(Bucket = "mhai-bk" , Prefix = "raw_data/authors/")
        files = [f"s3://mhai-bk/{item['Key']}" for item in response['Contents']]
        for file in files :
            authors = spark.read.format("parquet").load(file) 
            if "number_of_pages" in authors.columns :
                authors = authors.withColumn("number_of_pages" , col("number_of_pages").cast("long"))
            
            list_file.append(authors)
        authors = reduce(lambda a,b : a.unionByName(b , allowMissingColumns = True) , list_file)
        authors.limit(5).coalesce(1).write.mode("overwrite").parquet("s3://mhai-bk/data/authors/")

In [2]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("SkewExample") \
    .getOrCreate()

# Tạo RDD
rdd = spark.sparkContext.parallelize([
    ("VN", 1),
    ("VN", 2),
    ("US", 3),
    ("JP", 4)
])

# Convert sang DataFrame
df = rdd.toDF(["country", "id"])

df.show()

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.runJob.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 1.0 failed 1 times, most recent failure: Lost task 0.0 in stage 1.0 (TID 1) (MinhHai executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed). Consider setting 'spark.sql.execution.pyspark.udf.faulthandler.enabled' or'spark.python.worker.faulthandler.enabled' configuration to 'true' for the better Python traceback.
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:621)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:599)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:945)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.sca

In [1]:
import sys
import os

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("SkewExample").getOrCreate()

# Data skew
large_data = [("VN", i) for i in range(1000000)] + \
             [("US", i) for i in range(1000)] + \a aá
             [("JP", i) for i in range(1000)]

df = spark.createDataFrame(large_data, ["country", "id"])

In [ ]:
aádư'\

In [ ]:
df.show()

Py4JJavaError: An error occurred while calling o129.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 5.0 failed 1 times, most recent failure: Lost task 0.0 in stage 5.0 (TID 5) (MinhHai executor driver): java.io.IOException: Cannot run program "python3": CreateProcess error=2, The system cannot find the file specified
	at java.base/java.lang.ProcessBuilder.start(ProcessBuilder.java:1143)
	at java.base/java.lang.ProcessBuilder.start(ProcessBuilder.java:1073)
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:218)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:143)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:158)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:178)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:261)
	at org.apache.spark.api.python.PythonRDD.compute(PythonRDD.scala:70)


In [ ]:
result = df.groupBy("country").count()

result.show()

In [ ]:
df.show(5)

Py4JJavaError: An error occurred while calling o72.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 1.0 failed 1 times, most recent failure: Lost task 0.0 in stage 1.0 (TID 1) (MinhHai executor driver): java.io.IOException: Cannot run program "python3": CreateProcess error=2, The system cannot find the file specified
	at java.base/java.lang.ProcessBuilder.start(ProcessBuilder.java:1143)
	at java.base/java.lang.ProcessBuilder.start(ProcessBuilder.java:1073)
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:218)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:143)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:158)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:178)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:261)
	at org.apache.spark.api.python.PythonRDD.compute(PythonRDD.scala:70)
	

In [ ]:
salted_df = df.withColumn(
    "salt",
    (rand() * 10).cast("int")
)

salted_df = salted_df.withColumn(
    "country_salt",
    concat(col("country"), lit("_"), col("salt"))
)

partial = salted_df.groupBy("country_salt").count()

final = partial.withColumn(
    "country",
    split(col("country_salt"), "_")[0]
).groupBy("country").sum("count")

final.show()